<a href="https://colab.research.google.com/github/LtFordo/SCY1101_Precios_Consumidor/blob/main/ProyectoProgramaci%C3%B3nCienciaDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/LtFordo/SCY1101_Precios_Consumidor

fatal: destination path 'SCY1101_Precios_Consumidor' already exists and is not an empty directory.


In [ ]:
# Importación de las librerías a utilizar.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Se carga el dataset en su estado original para ejecutar posteriormente procesos de limpieza,
#validación y transformación de datos.
raw=pd.read_csv('SCY1101_Precios_Consumidor/data/precio_consumidor_2025_raw.csv')

raw['Region'].value_counts()


,count
Region,
Región Metropolitana de Santiago,69114
Región de La Araucanía,62762
Región de Valparaíso,58117
Región de Coquimbo,29576
Región del Biobío,28119
Región de Ñuble,27001
Región del Maule,25877
Región de Los Lagos,24960
Región de Arica y Parinacota,15622


In [ ]:
# Respaldo del dataset original para trabajar sin reparos.
df = raw.copy()

In [ ]:
# =================================================================
# FASE 1: AUDITORÍA DE INTEGRIDAD Y PROCEDENCIA
# =================================================================
# Justificación: La auditoría inicial permite caracterizar el dataset
# y detectar inconsistencias de tipos o esquemas antes de la manipulación.

try:
    print("--- Auditoría Inicial del Dataset ---")
    print(raw.info())
    # Verificación de valores imposibles y distribución inicial
    print("\n--- Estadísticas Descriptivas Crudas ---")
    print(raw.describe())
except Exception as e:
    print(f"Error en auditoría inicial: {e}")
df

--- Auditoría Inicial del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366618 entries, 0 to 366617
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Anio                     348367 non-null  float64
 1   Mes                      348716 non-null  float64
 2   Semana                   348563 non-null  float64
 3   Fecha inicio             348628 non-null  object 
 4   Fecha termino            348658 non-null  object 
 5   ID region                348217 non-null  float64
 6   Region                   348480 non-null  object 
 7   Sector                   348682 non-null  object 
 8   Tipo de punto monitoreo  348723 non-null  object 
 9   Grupo                    348650 non-null  object 
 10  Producto                 348662 non-null  object 
 11  Unidad                   348436 non-null  object 
 12  Precio minimo            348938 non-null  object 
 13  Precio maximo        

,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Precio_IVA
12,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Carnicería,Carne bovina,Palanca,$/kilo,12398.0,12998.0,12698.000000,15110.620000
123,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Supermercado,Abarrotes y otros,Spaghetti N°5,$/envase 400 gramos,520.0,1330.0,1015.384615,1208.307692
211,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Feria libre,Frutas,Limón|Sin especificar|2a amarillo,$/kilo,700.0,700.0,700.000000,833.000000
304,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Carne bovina,Posta Negra,$/kilo,9690.0,10490.0,10090.000000,12007.100000
354,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Hortalizas,Tomate|Larga vida|Primera,$/kilo,1950.0,1990.0,1963.333333,2336.366666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366327,2025.0,3.0,11.0,2025-03-10,2025-03-14,5.0,Región de Valparaíso,Viña del Mar,Supermercado,Carne de Cerdo - Ave - Cordero,Chuleta (Parrillera),$/kilo,4990.0,5790.0,5440.000000,6473.600000
366334,2025.0,12.0,50.0,2025-12-08,2025-12-12,5.0,Región de Valparaíso,Quillota,Supermercado,Carne bovina,Posta Negra,$/kilo,9990.0,11490.0,10923.333333,12998.766666
366358,2025.0,2.0,7.0,2025-02-10,2025-02-14,13.0,Región Metropolitana de Santiago,Oriente,Supermercado,Carne bovina,Lomo Vetado,No especificado,14990.0,17590.0,16061.428571,19113.099999
366407,2025.0,7.0,27.0,2025-06-30,2025-07-04,7.0,Región del Maule,Curicó,Supermercado,Abarrotes y otros,Azúcar,$/kilo,1190.0,1490.0,1330.000000,1582.700000


In [ ]:
# =================================================================
# FASE 2: LIMPIEZA PROFUNDA
# =================================================================
# Justificación: Se aplican múltiples técnicas de limpieza para eliminar
# el Bias (Sesgo) y asegurar que el modelo no procese datos inconsistentes.

try:
    # 1. Gestión de Duplicados: Eliminación de registros idénticos
    # que sesgan las métricas estadísticas.
    df.drop_duplicates(inplace=True)

    # 2. Corrección de Tipo de Dato: Conversión forzada a tipos numéricos y fechas.
    # Es mandatorio para permitir transformaciones avanzadas y cálculos matemáticos [7, 9].
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    # 3. Tratamiento de Outliers e Inconsistencias Lógicas:
    # Se eliminan valores imposibles (Mes > 12, Semana > 52) y precios negativos.
    df = df[(df['Mes'] <= 12) & (df['Semana'] <= 53)]
    df = df[df['Precio promedio'] >= 0]

    # Filtrado de outliers de error de digitación (Precios mínimos que exceden promedios)
    limite_max = df['Precio promedio'].max()
    df = df[df['Precio minimo'] <= limite_max]

    # 4. Normalización de Categóricos (Columna Region):
    # Eliminación de espacios y gestión de nulos representados como texto.
    df['Region'] = df['Region'].str.strip()
    valor_moda_region = df['Region'].mode()[0] # Extracción de valor escalar
    df['Region'] = df['Region'].replace('None', valor_moda_region)

    # Columna Sector
    df['Sector'] = (df['Sector'].replace('None', pd.NA))
    df['Sector'] = (df.groupby('Region')['Sector'].transform(lambda x: x.fillna(x.mode()[0])))


    # 5. Imputación Profesional de Nulos (Missing Values):
    # Se utiliza imputación lógica para no perder registros valiosos.
    # Precios faltantes se rellenan con el promedio del registro para mantener la integridad.
    df['Precio minimo'] = df['Precio minimo'].fillna(df['Precio promedio'])
    df['Precio maximo'] = df['Precio maximo'].fillna(df['Precio promedio'])

    # Categorías faltantes se etiquetan para auditoría posterior
    cols_cat = ['Sector', 'Tipo de punto monitoreo', 'Grupo', 'Unidad']
    df[cols_cat] = df[cols_cat].fillna('No especificado')

    # Barrido final para asegurar 0 nulos
    df.dropna(inplace=True)

    print("Fase de limpieza completada exitosamente.")

except Exception as e:
    print(f"Error crítico en el flujo de limpieza: {e}")
df

Fase de limpieza completada exitosamente.


,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Region,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Precio_IVA
12,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Carnicería,Carne bovina,Palanca,$/kilo,12398.0,12998.0,12698.000000,15110.620000
123,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,Coquimbo,Supermercado,Abarrotes y otros,Spaghetti N°5,$/envase 400 gramos,520.0,1330.0,1015.384615,1208.307692
211,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Feria libre,Frutas,Limón|Sin especificar|2a amarillo,$/kilo,700.0,700.0,700.000000,833.000000
304,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Carne bovina,Posta Negra,$/kilo,9690.0,10490.0,10090.000000,12007.100000
354,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Región de Coquimbo,La Serena,Supermercado,Hortalizas,Tomate|Larga vida|Primera,$/kilo,1950.0,1990.0,1963.333333,2336.366666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366327,2025.0,3.0,11.0,2025-03-10,2025-03-14,5.0,Región de Valparaíso,Viña del Mar,Supermercado,Carne de Cerdo - Ave - Cordero,Chuleta (Parrillera),$/kilo,4990.0,5790.0,5440.000000,6473.600000
366334,2025.0,12.0,50.0,2025-12-08,2025-12-12,5.0,Región de Valparaíso,Quillota,Supermercado,Carne bovina,Posta Negra,$/kilo,9990.0,11490.0,10923.333333,12998.766666
366358,2025.0,2.0,7.0,2025-02-10,2025-02-14,13.0,Región Metropolitana de Santiago,Oriente,Supermercado,Carne bovina,Lomo Vetado,No especificado,14990.0,17590.0,16061.428571,19113.099999
366407,2025.0,7.0,27.0,2025-06-30,2025-07-04,7.0,Región del Maule,Curicó,Supermercado,Abarrotes y otros,Azúcar,$/kilo,1190.0,1490.0,1330.000000,1582.700000


In [ ]:
# =================================================================
# FASE 3: TRANSFORMACIÓN AVANZADA
# =================================================================
# Justificación: Se utiliza vectorización y escalamiento para mejorar
# el poder predictivo y normalizar la varianza de los datos.

try:
    # 1. Ingeniería de Características: Creación de columna con IVA mediante broadcasting.
    # La vectorización optimiza la memoria al evitar ciclos.
    df['Precio_IVA'] = df['Precio promedio'] * 1.19

    # 2. Codificación Categórica (One-Hot Encoding):
    # Transforma 'Region' en variables binarias procesables por modelos de IA.
    df_final = pd.get_dummies(df, columns=['Region'], prefix='Reg', dtype=int)

    # 3. Estandarización (Scaling):
    # Uso de StandardScaler para ajustar los precios a una media 0 y desviación 1.
    # Justificación: Esta técnica asume una distribución gaussiana y mitiga
    # el impacto de la magnitud de los precios en el análisis.
    scaler = StandardScaler()
    df_final['Precio_Promedio_Escalado'] = scaler.fit_transform(df_final[['Precio promedio']])

    print("Fase de transformación avanzada finalizada.")

except Exception as e:
    print(f"Error en transformaciones avanzadas: {e}")

df_final[['ID region', 'Sector']].value_counts()



Fase de transformación avanzada finalizada.


ID region  Sector                
15.0       Arica                     173
13.0       Poniente                  137
           Sur                       136
5.0        Viña del Mar              136
9.0        Centro-Estación           135
                                    ... 
13.0         Mercado Lo Valledor       1
             Sur                       1
             None                      1
             Norte                     1
15.0         None                      1
Name: count, Length: 89, dtype: int64

In [ ]:
# =================================================================
# FASE 4: VALIDACIÓN Y EXPORTACIÓN
# =================================================================
# Justificación: La reproducibilidad exige que el archivo procesado se guarde
# en la estructura de carpetas definida para el proyecto (/data) [19-21].

try:
    print("--- Validación Final de Integridad ---")
    print(f"Registros totales: {len(df_final)}")
    print(f"Nulos remanentes: {df_final.isnull().sum().sum()}")

    # 1. Definición de la ruta de salida (Requisito de estructura de carpetas)
    # Se utiliza la ruta preexistente en la carpeta raíz para cumplir con
    # la organización jerárquica del repositorio (/data) [5, 6].
    ruta_salida = "SCY1101_Precios_Consumidor/data/precio_consumidor_2025_processed.csv"

    # 2. Exportación del Dataset Preparado
    # Justificación: El archivo se exporta en formato CSV sin índice para asegurar
    # la interoperabilidad y que el código corra "a la primera" en otros entornos [3, 7].
    df_final.to_csv(ruta_salida, index=False)

    print(f"Dataset final exportado exitosamente a: {ruta_salida}")

except Exception as e:
    # El manejo de excepciones es un requisito formal para garantizar la
    # robustez del flujo profesional de datos [1].
    print(f"Error en la fase final de validación o exportación: {e}")

try:
    print("--- Auditoría Inicial del Dataset ---")
    print(df_final.info())
    # Verificación de valores imposibles y distribución inicial
    print("\n--- Estadísticas Descriptivas Crudas ---")
    print(df_final.describe())
except Exception as e:
    print(f"Error en auditoría inicial: {e}")

df_final

--- Validación Final de Integridad ---
Registros totales: 3965
Nulos remanentes: 0
Dataset final exportado exitosamente a: SCY1101_Precios_Consumidor/data/precio_consumidor_2025_processed.csv
--- Auditoría Inicial del Dataset ---
<class 'pandas.core.frame.DataFrame'>
Index: 3965 entries, 12 to 366616
Data columns (total 25 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Anio                                  3965 non-null   float64       
 1   Mes                                   3965 non-null   float64       
 2   Semana                                3965 non-null   float64       
 3   Fecha inicio                          3965 non-null   datetime64[ns]
 4   Fecha termino                         3965 non-null   datetime64[ns]
 5   ID region                             3965 non-null   float64       
 6   Sector                                3965 non-null   object        

,Anio,Mes,Semana,Fecha inicio,Fecha termino,ID region,Sector,Tipo de punto monitoreo,Grupo,Producto,...,Reg_Región Metropolitana de Santiago,Reg_Región de Arica y Parinacota,Reg_Región de Coquimbo,Reg_Región de La Araucanía,Reg_Región de Los Lagos,Reg_Región de Valparaíso,Reg_Región de Ñuble,Reg_Región del Biobío,Reg_Región del Maule,Precio_Promedio_Escalado
12,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Coquimbo,Carnicería,Carne bovina,Palanca,...,0,0,1,0,0,0,0,0,0,1.623522
123,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,Coquimbo,Supermercado,Abarrotes y otros,Spaghetti N°5,...,0,0,1,0,0,0,0,0,0,-0.780483
211,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,La Serena,Feria libre,Frutas,Limón|Sin especificar|2a amarillo,...,0,0,1,0,0,0,0,0,0,-0.845382
304,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,La Serena,Supermercado,Carne bovina,Posta Negra,...,0,0,1,0,0,0,0,0,0,1.086857
354,2025.0,1.0,1.0,2024-12-30,2025-01-03,4.0,La Serena,Supermercado,Hortalizas,Tomate|Larga vida|Primera,...,0,0,1,0,0,0,0,0,0,-0.585418
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366327,2025.0,3.0,11.0,2025-03-10,2025-03-14,5.0,Viña del Mar,Supermercado,Carne de Cerdo - Ave - Cordero,Chuleta (Parrillera),...,0,0,0,0,0,1,0,0,0,0.129998
366334,2025.0,12.0,50.0,2025-12-08,2025-12-12,5.0,Quillota,Supermercado,Carne bovina,Posta Negra,...,0,0,0,0,0,1,0,0,0,1.258338
366358,2025.0,2.0,7.0,2025-02-10,2025-02-14,13.0,Oriente,Supermercado,Carne bovina,Lomo Vetado,...,1,0,0,0,0,0,0,0,0,2.315636
366407,2025.0,7.0,27.0,2025-06-30,2025-07-04,7.0,Curicó,Supermercado,Abarrotes y otros,Azúcar,...,0,0,0,0,0,0,0,0,1,-0.715743


In [ ]:
try:
    # 1. Eliminación de Duplicados: Evita sesgar las estadísticas del modelo [6, 9]
    df.drop_duplicates(inplace=True)

    # 2. Corrección de tipos: Conversión a numérico y datetime [10]
    # Usamos errors='coerce' para gestionar datos mal formateados como NaN
    cols_precios = ['Precio minimo', 'Precio maximo', 'Precio promedio']
    for col in cols_precios:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Fecha inicio'] = pd.to_datetime(df['Fecha inicio'], errors='coerce')
    df['Fecha termino'] = pd.to_datetime(df['Fecha termino'], errors='coerce')

    print("Corrección de formatos y eliminación de duplicados finalizada.")
except Exception as e:
    print(f"Error en corrección de tipos: {e}")